# Bin Particles for Testing

In [ ]:
import pandas as pd
import numpy as np

INPUT_CSV  = #######
OUTPUT_CSV = #######
BIN_WIDTH  = 0.1    # µm
SAMPLE_N   = 30     # per bin

# ── Load & compute diameter in µm ────────────────────────────
df = pd.read_csv(INPUT_CSV)
if "diameter_m" in df.columns:
    df["diam_um"] = df["diameter_m"] * 1e6
else:
    df["diam_um"] = np.sqrt(df["major_m"] * df["minor_m"]) * 1e6

df = df.sort_values("diam_um").reset_index(drop=True)
df["bin"] = (np.floor(df["diam_um"] / BIN_WIDTH) * BIN_WIDTH).round(3)

def pick_evenly(g, n=SAMPLE_N):
    m = len(g)
    if m <= n:
        return g
    idx = np.floor(np.linspace(0, m-1, n)).astype(int)
    return g.iloc[idx]

selected = (
    df.groupby("bin", group_keys=False)
      .apply(lambda g: pick_evenly(g, SAMPLE_N))
      .reset_index(drop=True)     # throw away the old index entirely
      .drop(columns="bin")
)

# Now `selected` still has your original `ParticleID` column (and all the others),
# and no extra index column.
selected.to_csv(OUTPUT_CSV, index=False)

print(f"Picked {len(selected)} particles; each bin now has up to {SAMPLE_N} spread from min to max.")


# Final Full Loop

In [ ]:
"""
Autonomous Laser Ignition Testing Loop
-------------------------------------
This script requires:
- AMC piezo stage controller (custom API, not included)
- Siglent SDG waveform generator (PyVISA)
- Lumenera camera software (BlankCamera.exe)
- Windows GUI automation via pyautogui

The code is provided as-is to document the workflow used in the manuscript.
"""

import sys
# Example:
# sys.path.append("/path/to/local/modules")

import os
import time
import csv
import subprocess
import logging
import math
import uuid

import numpy as np
import pandas as pd
import pyautogui
import cv2
from PIL import Image, ImageDraw

import AMC
import pyvisa

# ── CONFIG ────────────────────────────────────────────────────────────────────
OFFSET_CSV       = r"laser_calibration_offset.csv"
RESULTS_CSV      = os.path.join(OUTPUT_DIR, "results.csv")
AMC_IP           = ######
SIGLENT_ADDR     = ######
CSV_PATH = "path/to/particle_database.csv"   # user must provide
OUTPUT_DIR = "results/"                      # user must provide
AMC_IP = "xxx.xxx.xxx.xxx"                   # instrument IP
SIGLENT_ADDR = "USB0::0xF4EC::0x1101::INSTR" # PyVISA resource

# NOTE: User should insert path to local modules or camera utility here.
# Example:
# sys.path.append("/path/to/local/modules")
BLANKCAMERA_EXE = "/path/to/BlankCamera.exe"

CONNECT_BUTTON_COORDS = (874, 412)
CAPTURE_BUTTON_COORDS = (1205, 634)
EXIT_CAPTURE_COORDS   = (1558, 13)
SAVE_BUTTON_COORDS    = (834, 563)
FILENAME_FIELD_COORDS = (700, 450)
EXIT_BUTTON_COORDS    = (1246, 366)

LEFT, UPPER   = 580, 600
WIDTH, HEIGHT = 211, 211
CROP_BOX      = (LEFT, UPPER, LEFT+WIDTH, UPPER+HEIGHT)
ROI_CENTER    = (WIDTH/2, HEIGHT/2)

UM_PER_PX_X   = 0.217
UM_PER_PX_Y   = 0.217
UM_PER_PX = (UM_PER_PX_X + UM_PER_PX_Y) / 2.0
TOLERANCE_XY  = 0.2     # µm
MAX_ITERS     = 1

PULSE_AMP_VPP  = 1.0
PULSE_PER_S    = 20e-6
PULSE_WIDTH_S  = 3e-6
PULSE_GAP_S    = 0.002
SDG_LOAD_MODE  = "INF"

# ── Hough (standard pass) ─────────────────────────────────────────────────────
HOUGH_DP             = 1.0
HOUGH_PARAM1         = 100
HOUGH_PARAM2_START   = 7
HOUGH_PARAM2_END     = 22
HOUGH_PARAM2_STEP    = 1
HOUGH_MIN_DIST_FRAC  = 0.5
RAD_TOL_FRAC         = 0.05
MIN_RADIUS_PIX       = 1
MAX_RADIUS_PIX       = 10

# ── Tiny-pass detector settings (upscaled) ────────────────────────────────────
MIN_DIAM_UM   = 0.4
MAX_DIAM_UM   = 3.5
UPSCALE       = 3

HOUGH_TINY_DP         = 1.0
HOUGH_TINY_PARAM1     = 100
HOUGH_TINY_PARAM2_MIN = 8
HOUGH_TINY_PARAM2_MAX = 16
HOUGH_TINY_PARAM2_STEP= 2
MIN_DIST_FACTOR_TINY  = 3.0

USE_CLAHE         = True
CLAHE_CLIP        = 3.0
GAUSS_BLUR_KSIZE  = 0
MEDIAN_BLUR_KSIZE = 3

CONF_THRESH   = 0.71
MIN_SEP_UM    = 4.0

# ── Diameter tolerance (+ scoring) ────────────────────────────────────────────
DIAM_TOL_ABS_UM = 0.12   # ±0.25 µm around expected; set to None to disable
DIAM_TOL_FRAC   = None   # or a fraction (e.g., 0.25 for ±25%); set to None to disable
W_SIZE_ERR      = 0.85   # weight on |d_meas - d_exp|/d_exp
W_CONF_TERM     = 0.15   # weight on (1 - confidence)

# ── Autofocus ─────────────────────────────────────────────────────────────────
Z_AXIS = 2
AF_DIST_THRESH_UM = 1500.0
AF_SETTLE_S        = 0.12
AF_COARSE_STEP_NM  = 500
AF_MAX_STEPS       = 6
AF_IMPROVE_EPS     = 0.05
AF_REL_DROP_STOP   = 0.12
AF_FINE_WINDOW_NM  = 1000
AF_FINE_STEP_NM    = 200

os.makedirs(OUTPUT_DIR, exist_ok=True)

def setup_logging():
    log_file = os.path.join(OUTPUT_DIR, "run.log")
    fh = logging.FileHandler(log_file, encoding="utf-8")
    sh = logging.StreamHandler()
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s %(levelname)s: %(message)s",
        handlers=[fh, sh]
    )
    return logging.getLogger()

logger = setup_logging()

# ── I/O + hardware helpers ───────────────────────────────────────────────────
def load_data():
    df = pd.read_csv(CSV_PATH)
    df["orig_idx"] = df["ParticleID"].astype(int)
    if "diameter_m" in df.columns:
        df["diam_um"] = df["diameter_m"] * 1e6
    else:
        df["diam_um"] = np.sqrt(df["major_m"] * df["minor_m"]) * 1e6
    df = df.sort_values("diam_um").reset_index(drop=True)

    offset = pd.read_csv(OFFSET_CSV)
    dx_nm = int((offset.loc[offset.Name=="Laser Spot","X_m"].iat[0] -
                 offset.loc[offset.Name=="Particle Center","X_m"].iat[0]) * 1e9)
    dy_nm = int((offset.loc[offset.Name=="Laser Spot","Y_m"].iat[0] -
                 offset.loc[offset.Name=="Particle Center","Y_m"].iat[0]) * 1e9)
    return df, dx_nm, dy_nm

def connect_devices():
    dev = AMC.Device(AMC_IP)
    dev.connect()
    dev.control.setControlMove(0, False)
    dev.control.setControlMove(1, False)
    rm = pyvisa.ResourceManager()
    sig = rm.open_resource(SIGLENT_ADDR)
    sig.read_termination  = '\n'
    sig.write_termination = '\n'
    return dev, rm, sig

def setup_sdg(inst, amp=PULSE_AMP_VPP, per=PULSE_PER_S, width=PULSE_WIDTH_S, load_mode=SDG_LOAD_MODE):
    inst.write("*CLS")
    inst.write("C1:OUTP OFF")
    inst.write("C1:BSWV WVTP,PULSE")
    inst.write(f"C1:BSWV PER,{per}")
    inst.write(f"C1:BSWV WIDTH,{width}")
    inst.write(f"C1:BSWV AMP,{amp}")
    inst.write("C1:BSWV OFST,0")
    inst.write("C1:BTWV:STATE OFF")
    inst.write(f"C1:OUTP LOAD,{load_mode}")

def fire_n_pulses(inst, n: int, gap_s: float = PULSE_GAP_S):
    if n <= 0:
        return
    inst.write("C1:OUTP ON")
    time.sleep(0.02)
    for _ in range(int(n)):
        inst.write("C1:TRIG")
        time.sleep(gap_s)
    inst.write("C1:OUTP OFF")

def get_axis_state(dev, axis: int) -> str:
    s = dev.Status(dev)
    _, in_t = s.getStatusTargetRange(axis)
    _, mv   = s.getStatusMoving(axis)
    if mv:   return "Moving"
    if in_t: return "In Target Range"
    return "Ready"

READY_STATES = {"Ready", "In Target Range"}

def wait_for_ready_or_stop(dev, axis, timeout=2, poll_interval=0.1):
    t0 = time.time()
    while True:
        state = get_axis_state(dev, axis)
        if state in READY_STATES:
            return True
        if time.time() - t0 > timeout:
            try: dev.control.setControlMove(axis, False)
            except Exception: pass
            return False
        time.sleep(poll_interval)

def move_and_wait(dev, axis, target):
    dev.move.setControlTargetPosition(axis, int(target))
    dev.control.setControlMove(axis, True)
    wait_for_ready_or_stop(dev, axis)

def _snap_crop_gray_quiet() -> np.ndarray:
    tmp_path = os.path.join(OUTPUT_DIR, f"__af_{uuid.uuid4().hex}.png")
    proc = subprocess.Popen([BLANKCAMERA_EXE])
    try:
        time.sleep(1.2)
        pyautogui.click(*CONNECT_BUTTON_COORDS); time.sleep(0.8)
        pyautogui.click(*CAPTURE_BUTTON_COORDS); time.sleep(1.2)
        pyautogui.click(*EXIT_CAPTURE_COORDS);   time.sleep(1.0)
        pyautogui.click(*SAVE_BUTTON_COORDS);    time.sleep(0.8)
        pyautogui.click(*FILENAME_FIELD_COORDS); time.sleep(0.8)
        pyautogui.hotkey('ctrl','a'); pyautogui.write(tmp_path)
        time.sleep(0.2); pyautogui.press('enter'); time.sleep(0.6)
        pyautogui.click(*EXIT_BUTTON_COORDS)
    finally:
        try: proc.terminate()
        except Exception: pass

    img = Image.open(tmp_path).crop(CROP_BOX).convert("L")
    arr = np.array(img, dtype=np.uint8)
    img.close()
    try: os.remove(tmp_path)
    except Exception: pass
    return arr

def snap_crop(orig_idx, tag, draw_circle=None):
    path = os.path.join(OUTPUT_DIR, f"p{orig_idx}_{tag}.png")
    proc = subprocess.Popen([BLANKCAMERA_EXE])
    time.sleep(1.5)
    pyautogui.click(*CONNECT_BUTTON_COORDS); time.sleep(0.8)
    pyautogui.click(*CAPTURE_BUTTON_COORDS); time.sleep(1.2)
    pyautogui.click(*EXIT_CAPTURE_COORDS);   time.sleep(1.0)
    pyautogui.click(*SAVE_BUTTON_COORDS);    time.sleep(0.8)
    pyautogui.click(*FILENAME_FIELD_COORDS)
    pyautogui.hotkey('ctrl','a'); pyautogui.write(path)
    time.sleep(0.8); pyautogui.press('enter'); time.sleep(0.8)
    pyautogui.click(*EXIT_BUTTON_COORDS)
    proc.terminate()

    img = Image.open(path).crop(CROP_BOX).convert("L")
    if draw_circle:
        img_rgb = img.convert("RGB")
        d = ImageDraw.Draw(img_rgb)
        x,y,r = draw_circle
        d.ellipse([x-r,y-r,x+r,y+r], outline="green", width=2)
        img_rgb.save(path)
        return np.array(img_rgb.convert("L"))
    img.save(path)
    return np.array(img, dtype=np.uint8)

# ── Autofocus routines ────────────────────────────────────────────────────────
def _focus_score(gray: np.ndarray) -> float:
    lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
    return float(lap.var())

def _measure_focus_at(dev, z_nm: int) -> float:
    move_and_wait(dev, Z_AXIS, int(z_nm))
    time.sleep(AF_SETTLE_S)
    return _focus_score(_snap_crop_gray_quiet())

def _hill_climb_quick(dev, z0: int) -> int:
    s0 = _measure_focus_at(dev, z0)
    s_plus  = _measure_focus_at(dev, z0 + AF_COARSE_STEP_NM)
    s_minus = _measure_focus_at(dev, z0 - AF_COARSE_STEP_NM)
    if max(s_plus, s_minus) <= s0 * (1 + AF_IMPROVE_EPS):
        step = AF_COARSE_STEP_NM * 2
        sp2  = _measure_focus_at(dev, z0 + step)
        sm2  = _measure_focus_at(dev, z0 - step)
        direction = +1 if sp2 >= sm2 else -1
        z_curr = z0 + direction * step
        best_z, best_s = (z0 + step, sp2) if direction == +1 else (z0 - step, sm2)
        best_z = int(best_z)
    else:
        direction = +1 if s_plus >= s_minus else -1
        z_curr = z0 + direction * AF_COARSE_STEP_NM
        if direction == +1:
            best_z, best_s = int(z0 + AF_COARSE_STEP_NM), s_plus
        else:
            best_z, best_s = int(z0 - AF_COARSE_STEP_NM), s_minus

    steps = 0
    while steps < AF_MAX_STEPS:
        z_next = z_curr + direction * AF_COARSE_STEP_NM
        s_next = _measure_focus_at(dev, z_next)
        steps += 1
        if s_next > best_s * (1 + AF_IMPROVE_EPS):
            best_s, best_z = s_next, int(z_next)
        else:
            if (best_s - s_next) / max(best_s, 1e-9) >= AF_REL_DROP_STOP:
                break
        z_curr = z_next
    return int(best_z)

def _refine_local_quick(dev, z_best: int) -> int:
    start = int(z_best - AF_FINE_WINDOW_NM)
    stop  = int(z_best + AF_FINE_WINDOW_NM)
    best_z, best_s = int(z_best), -1.0
    prev_s = None
    down = 0
    for z in range(start, stop + 1, AF_FINE_STEP_NM):
        s = _measure_focus_at(dev, z)
        if s > best_s:
            best_s, best_z = s, int(z)
        if prev_s is not None and s < prev_s:
            down += 1
            if down >= 2 and (best_s - s) / max(best_s, 1e-9) >= 0.10:
                break
        else:
            down = 0
        prev_s = s
    move_and_wait(dev, Z_AXIS, best_z)
    return int(best_z)

def fast_autofocus_quiet(dev, do_refine=True):
    try:
        z0 = int(dev.move.getPosition(Z_AXIS)[1])
        z_peak = _hill_climb_quick(dev, z0)
        if do_refine:
            z_peak = _refine_local_quick(dev, z_peak)
        move_and_wait(dev, Z_AXIS, z_peak)
    except Exception:
        pass

def maybe_autofocus_after_large_move(dev, x_target_nm: int, y_target_nm: int, threshold_um: float = AF_DIST_THRESH_UM):
    curx = dev.move.getPosition(0)[1]
    cury = dev.move.getPosition(1)[1]
    dist_um = ( (x_target_nm - curx)**2 + (y_target_nm - cury)**2 )**0.5 / 1e3
    move_and_wait(dev, 0, int(x_target_nm))
    move_and_wait(dev, 1, int(y_target_nm))
    if dist_um >= threshold_um:
        fast_autofocus_quiet(dev, do_refine=True)

# ── Circle detection (standard + tiny) ────────────────────────────────────────
class NoCircleWithinTolerance(Exception):
    pass

def _preprocess_standard(gray: np.ndarray) -> np.ndarray:
    g = gray.copy()
    g = cv2.medianBlur(g, 3)
    g = cv2.GaussianBlur(g, (7, 7), 0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(g)

def _detect_circles_standard(gray: np.ndarray) -> np.ndarray:
    min_r = max(2, int(round((0.2 / 2.0) / UM_PER_PX)))
    max_r = max(min_r + 1, int(round((2.00 / 2.0) / UM_PER_PX)))
    min_dist = max(2, int(min_r * HOUGH_MIN_DIST_FRAC))
    for p2 in range(HOUGH_PARAM2_START, HOUGH_PARAM2_END + 1, HOUGH_PARAM2_STEP):
        circ = cv2.HoughCircles(
            gray, cv2.HOUGH_GRADIENT, dp=HOUGH_DP,
            minDist=min_dist, param1=HOUGH_PARAM1, param2=p2,
            minRadius=min_r, maxRadius=max_r
        )
        if circ is not None and len(circ[0]) > 0:
            return np.around(circ[0]).astype(int)
    return np.empty((0,3), dtype=int)

def preprocess_tiny(gray):
    g = gray.copy()
    if MEDIAN_BLUR_KSIZE and (MEDIAN_BLUR_KSIZE % 2 == 1):
        g = cv2.medianBlur(g, MEDIAN_BLUR_KSIZE)
    if GAUSS_BLUR_KSIZE and (GAUSS_BLUR_KSIZE % 2 == 1):
        g = cv2.GaussianBlur(g, (GAUSS_BLUR_KSIZE, GAUSS_BLUR_KSIZE), 0)
    if USE_CLAHE:
        clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8,8))
        g = clahe.apply(g)
    return g

def detect_circles_tiny(gray_pre):
    h, w = gray_pre.shape[:2]
    big = cv2.resize(gray_pre, (w*UPSCALE, h*UPSCALE), interpolation=cv2.INTER_CUBIC)
    min_r_orig = (MIN_DIAM_UM / 2.0) / UM_PER_PX
    max_r_orig = (MAX_DIAM_UM / 2.0) / UM_PER_PX
    min_r = max(1, int(np.floor(min_r_orig * UPSCALE)))
    max_r = max(min_r + 1, int(np.ceil (max_r_orig * UPSCALE)))
    min_dist = max(2, int(min_r * MIN_DIST_FACTOR_TINY))
    found = None
    for p2 in range(HOUGH_TINY_PARAM2_MIN, HOUGH_TINY_PARAM2_MAX + 1, HOUGH_TINY_PARAM2_STEP):
        circles = cv2.HoughCircles(
            big, cv2.HOUGH_GRADIENT, dp=HOUGH_TINY_DP,
            minDist=min_dist, param1=HOUGH_TINY_PARAM1, param2=p2,
            minRadius=min_r, maxRadius=max_r
        )
        if circles is not None and len(circles[0]) > 0:
            found = circles[0]
            break
    if found is None:
        return np.empty((0,3), dtype=float)
    mapped = []
    for (cx, cy, r) in found:
        mapped.append([cx / UPSCALE, cy / UPSCALE, r / UPSCALE])
    return np.array(mapped, dtype=float)

def _edge_fraction_on_circle(img, cx, cy, r, canny_low=40, canny_high=HOUGH_PARAM1, tol_px=1.5):
    edges = cv2.Canny(img, canny_low, canny_high)
    if tol_px > 0:
        k = int(max(1, round(tol_px)))
        edges = cv2.dilate(edges, np.ones((2*k+1, 2*k+1), np.uint8), iterations=1)
    npts = max(24, int(2*math.pi*r/2))
    ang  = np.linspace(0, 2*np.pi, npts, endpoint=False)
    xs   = (cx + r*np.cos(ang)).astype(int)
    ys   = (cy + r*np.sin(ang)).astype(int)
    h, w = img.shape
    m = (xs>=0)&(xs<w)&(ys>=0)&(ys<h)
    if not m.any(): return 0.0
    return (edges[ys[m], xs[m]] > 0).mean()

def _ring_samples(img, cx, cy, r, ring_w=2):
    h, w = img.shape
    yy, xx = np.ogrid[:h, :w]
    dist = np.sqrt((xx - cx)**2 + (yy - cy)**2)
    inner = (dist >= r - 2*ring_w) & (dist <= r - ring_w)
    edge  = (dist >= r - ring_w)  & (dist <= r + ring_w)
    outer = (dist >= r + ring_w)  & (dist <= r + 2*ring_w)
    if not (inner.any() and edge.any() and outer.any()):
        return None, None, None
    return img[inner].mean(), img[edge].mean(), img[outer].mean()

def confidence_score(img_pre, cx, cy, r):
    ef = _edge_fraction_on_circle(img_pre, cx, cy, int(round(r)))
    i, e, o = _ring_samples(img_pre, cx, cy, int(round(r)), ring_w=2)
    rc = 0.0 if i is None else max(0.0, (e - 0.5*(i+o))) / 255.0
    raw = 0.75*ef + 0.25*rc
    return float(1.0 / (1.0 + np.exp(-6.0*(raw - 0.6))))

def _bilinear_sample(img, xs, ys):
    h, w = img.shape
    x0 = np.floor(xs).astype(int); x1 = x0 + 1
    y0 = np.floor(ys).astype(int); y1 = y0 + 1
    x0 = np.clip(x0, 0, w-1); x1 = np.clip(x1, 0, w-1)
    y0 = np.clip(y0, 0, h-1); y1 = np.clip(y1, 0, h-1)
    Ia = img[y0, x0]; Ib = img[y1, x0]; Ic = img[y0, x1]; Id = img[y1, x1]
    wa = (x1 - xs) * (y1 - ys)
    wb = (x1 - xs) * (ys - y0)
    wc = (xs - x0) * (y1 - ys)
    wd = (xs - x0) * (ys - y0)
    return Ia*wa + Ib*wb + Ic*wc + Id*wd

def _quadratic_subpixel(x, y, i):
    if i <= 0 or i >= len(x)-1:
        return x[i], y[i]
    x1,x2,x3 = x[i-1], x[i], x[i+1]
    y1,y2,y3 = y[i-1], y[i], y[i+1]
    denom = (y1 - 2*y2 + y3)
    if abs(denom) < 1e-6:
        return x2, y2
    x_peak = x2 + 0.5 * (y1 - y3) / denom * (x3 - x1)/2.0
    return x_peak, y2

def refine_radius_radial_gradient(img_pre, cx, cy, r_init,
                                  ray_count=64, r_span_frac=0.5, samples=32):
    r_min = max(1.0, r_init * (1.0 - r_span_frac))
    r_max = max(r_min + 1.0, r_init * (1.0 + r_span_frac))
    rs = np.linspace(r_min, r_max, samples)
    dr = rs[1] - rs[0]
    thetas = np.linspace(0, 2*np.pi, ray_count, endpoint=False)
    peaks = []
    for th in thetas:
        xs = cx + rs * np.cos(th)
        ys = cy + rs * np.sin(th)
        prof = _bilinear_sample(img_pre, xs, ys)
        dI = np.gradient(prof, dr)
        k = int(np.argmax(np.abs(dI)))
        r_peak, _ = _quadratic_subpixel(rs, np.abs(dI), k)
        peaks.append(float(r_peak))
    r_ref = float(np.median(peaks))
    r_ref = float(np.clip(r_ref, r_min, r_max))
    return r_ref

def _enforce_min_spacing(rows, min_sep_px):
    rows = sorted(rows, key=lambda d: d["confidence"], reverse=True)
    kept = []
    for d in rows:
        ok = True
        for k in kept:
            if (d["x_px"]-k["x_px"])**2 + (d["y_px"]-k["y_px"])**2 < (min_sep_px**2):
                ok = False; break
        if ok: kept.append(d)
    return kept

def detect_circle_v2(gray_roi_u8: np.ndarray, expected_d_px: float):
    """Return (cx, cy, r_px, diam_um, confidence).
       Raises RuntimeError (no circles) or NoCircleWithinTolerance (out of tol)."""
    # standard
    g_std = _preprocess_standard(gray_roi_u8)
    cand_std = _detect_circles_standard(g_std)
    std_rows = []
    for (cx, cy, r) in cand_std:
        diam_um = (2*r) * UM_PER_PX
        conf = confidence_score(g_std, int(cx), int(cy), r)
        std_rows.append(dict(x_px=float(cx), y_px=float(cy), r_px=float(r),
                             diam_um=float(diam_um), confidence=float(conf), src="std"))

    # tiny
    g_tiny = preprocess_tiny(gray_roi_u8)
    cand_tiny = detect_circles_tiny(g_tiny)
    tiny_rows = []
    for (cx, cy, r) in cand_tiny:
        diam_um = (2*r) * UM_PER_PX
        conf = confidence_score(g_tiny, int(round(cx)), int(round(cy)), r)
        tiny_rows.append(dict(x_px=float(cx), y_px=float(cy), r_px=float(r),
                              diam_um=float(diam_um), confidence=float(conf), src="tiny"))

    rows = std_rows + tiny_rows
    if not rows:
        raise RuntimeError("No circles detected in ROI (std+tiny).")

    # prefilter: confidence + spacing
    min_sep_px = int(round(MIN_SEP_UM / UM_PER_PX))
    rows = [r for r in rows if r["confidence"] >= CONF_THRESH]
    if min_sep_px > 0 and rows:
        rows = _enforce_min_spacing(rows, min_sep_px)
    if not rows:
        raise RuntimeError("Detections filtered out by confidence/spacing.")

    # tolerance gate
    d_exp_um = expected_d_px * UM_PER_PX
    def within_tol(d_um: float) -> bool:
        ok_abs  = (DIAM_TOL_ABS_UM is not None) and (abs(d_um - d_exp_um) <= DIAM_TOL_ABS_UM)
        ok_frac = (DIAM_TOL_FRAC   is not None) and (abs(d_um - d_exp_um) <= DIAM_TOL_FRAC * max(d_exp_um, 1e-9))
        if (DIAM_TOL_ABS_UM is None) and (DIAM_TOL_FRAC is None):
            return True
        return ok_abs or ok_frac

    cand_in = [r for r in rows if within_tol(r["diam_um"])]
    if not cand_in:
        raise NoCircleWithinTolerance(f"No circles within tolerance around {d_exp_um:.3f} µm")

    # score
    def score(r):
        size_err  = abs(r["diam_um"] - d_exp_um) / max(d_exp_um, 1e-9)
        conf_term = 1.0 - r["confidence"]
        return W_SIZE_ERR*size_err + W_CONF_TERM*conf_term

    best = min(cand_in, key=score)

    # optional refine
    base_img = g_tiny if best["src"]=="tiny" else g_std
    try:
        r_ref = refine_radius_radial_gradient(base_img, best["x_px"], best["y_px"], best["r_px"],
                                              ray_count=64, samples=32)
    except Exception:
        r_ref = best["r_px"]

    cx_i, cy_i, r_i = int(round(best["x_px"])), int(round(best["y_px"])), int(round(r_ref))
    diam_um = float((2*r_ref)*UM_PER_PX)
    conf    = float(best["confidence"])
    return cx_i, cy_i, r_i, diam_um, conf

# ── Explosion difference metric ───────────────────────────────────────────────
def difference_metric_circle(
    before: np.ndarray,
    after:  np.ndarray,
    center,
    radius: int,
    *,
    pad_px: int = None,
    win_scale: float = 2.5,
    k_sigma: float = 3.0,
    min_delta: int = 10,
    frac_core: float = 0.20,
    frac_window: float = 0.05,
    frac_global: float = 0.02
) -> bool:
    h, w = before.shape[:2]
    cx, cy = int(center[0]), int(center[1])
    r      = int(max(1, radius))
    diff = cv2.absdiff(before, after).astype(np.uint8)
    med  = np.median(diff)
    mad  = np.median(np.abs(diff - med))
    thr  = max(min_delta, int(med + k_sigma * 1.4826 * (mad + 1e-6)))

    changed = (diff > thr).astype(np.uint8) * 255
    k = np.ones((3,3), np.uint8)
    changed = cv2.morphologyEx(changed, cv2.MORPH_OPEN,  k)
    changed = cv2.morphologyEx(changed, cv2.MORPH_CLOSE, k)
    changed_bool = changed.astype(bool)

    if pad_px is None:
        pad_px = max(5, int(round(0.35 * r)))
    R_core = r + pad_px

    Y, X = np.ogrid[:h, :w]
    dist2   = (X - cx)**2 + (Y - cy)**2
    core_m  = dist2 <= (R_core * R_core)

    half = int(max(R_core, win_scale * r))
    x0, x1 = max(0, cx - half), min(w, cx + half + 1)
    y0, y1 = max(0, cy - half), min(h, cy + half + 1)
    window_m = np.zeros_like(changed_bool, dtype=bool)
    window_m[y0:y1, x0:x1] = True

    global_m = np.ones_like(changed_bool, dtype=bool)

    def frac(mask):
        cnt = mask.sum()
        return float(changed_bool[mask].sum()) / max(cnt, 1)

    f_core   = frac(core_m)
    f_win    = frac(window_m)
    f_global = frac(global_m)

    return (f_core >= frac_core) or (f_win >= frac_window) or (f_global >= frac_global)

# ── Alignment using the pluggable detector ───────────────────────────────────
def align_particle(dev, row, orig_idx, expected_d_px):
    # 1) move + autofocus gate
    x_nm = int(row["lm_stage_x_m"] * 1e9)
    y_nm = int(row["lm_stage_y_m"] * 1e9)
    maybe_autofocus_after_large_move(dev, x_nm, y_nm, threshold_um=AF_DIST_THRESH_UM)

    # 2) fine alignment loop
    cx = cy = r = None
    for i in range(MAX_ITERS):
        img = snap_crop(orig_idx, f"align{i}")
        try:
            cx, cy, r, d_um, conf = detect_circle_v2(img, expected_d_px)
        except NoCircleWithinTolerance as e:
            raise  # let caller handle + log "skipped"
        except Exception as e:
            # no circles at all / filtered out — try next iteration (fresh snap)
            if i == MAX_ITERS - 1:
                raise

        dx_um = (cx-ROI_CENTER[0]) * UM_PER_PX_X
        dy_um = (cy-ROI_CENTER[1]) * UM_PER_PX_Y
        logger.info(f"[p{orig_idx}] det: d={d_um:.3f} µm, conf={conf:.2f}, offs=({dx_um:+.3f},{dy_um:+.3f}) µm")
        if abs(dx_um) < TOLERANCE_XY and abs(dy_um) < TOLERANCE_XY:
            break
        curx = dev.move.getPosition(0)[1]
        cury = dev.move.getPosition(1)[1]
        move_and_wait(dev, 0, curx + int(np.sign(dx_um)*abs(dx_um)*1e3))
        move_and_wait(dev, 1, cury - int(np.sign(dy_um)*abs(dy_um)*1e3))
        time.sleep(0.5)
        fn = os.path.join(OUTPUT_DIR, f"p{orig_idx}_align{i}.png")
        try: os.remove(fn)
        except FileNotFoundError: pass

    aligned_x = dev.move.getPosition(0)[1]
    aligned_y = dev.move.getPosition(1)[1]
    before = snap_crop(orig_idx, "before", draw_circle=(cx, cy, r))
    return aligned_x, aligned_y, before, (cx, cy, r)

# ── Results logging (now with measured_diam_um + status) ─────────────────────
def log_results(orig_idx, aligned_x, aligned_y, exploded, cycles,
                diameter_um, duration_sec, measured_diam_um=None, status="done"):
    header = not os.path.exists(RESULTS_CSV)
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.writer(f)
        if header:
            w.writerow(["particle_id","aligned_x_nm","aligned_y_nm",
                        "exploded","cycle_count","diameter_um","duration_sec",
                        "measured_diam_um","status"])
        w.writerow([
            orig_idx,
            "" if aligned_x is None else aligned_x,
            "" if aligned_y is None else aligned_y,
            exploded,
            cycles,
            round(diameter_um,2) if diameter_um is not None else "",
            round(duration_sec, 2) if duration_sec is not None else "",
            "" if measured_diam_um is None else round(measured_diam_um, 3),
            status
        ])
    logger.info(f"Particle {orig_idx} -> status={status}, exploded={exploded}, cycles={cycles}")

# ── Main loop ────────────────────────────────────────────────────────────────
def main():
    df, dx_nm, dy_nm = load_data()

    # Skip any particle_id already present in results (including 'skipped')
    if os.path.exists(RESULTS_CSV):
        done = pd.read_csv(RESULTS_CSV)['particle_id'].astype(int)
        df = df[~df.orig_idx.isin(done)].reset_index(drop=True)
        logger.info(f"Skipping {len(done)} completed/skipped particles")

    # resume cycle_count
    if os.path.exists(RESULTS_CSV):
        prev = pd.read_csv(RESULTS_CSV)
        cycle_count = int(prev['cycle_count'].max())
        logger.info(f"Resuming with cycle_count={cycle_count}")
    else:
        cycle_count = 2

    no_explosion_streak = 0

    dev, rm, sig = connect_devices()
    setup_sdg(sig, amp=PULSE_AMP_VPP, per=PULSE_PER_S, width=PULSE_WIDTH_S)

    try:
        for _, row in df.iterrows():
            orig_idx   = int(row["orig_idx"])
            diameter_um= float(row["diam_um"])
            exp_d_px = diameter_um / UM_PER_PX
            start_time = time.time()
            time.sleep(3)

            try:
                aligned_x, aligned_y, before, circle = align_particle(
                    dev, row, orig_idx, exp_d_px
                )
            except NoCircleWithinTolerance as e:
                # explicitly log as "skipped" so resume ignores it
                logger.info(f"[p{orig_idx}] {e}; marking as skipped.")
                log_results(orig_idx, None, None, False, cycle_count,
                            diameter_um, 0.0, measured_diam_um=None, status="skipped")
                continue
            except Exception as e:
                logger.warning(f"[p{orig_idx}] alignment failed: {e}; marking as skipped.")
                log_results(orig_idx, None, None, False, cycle_count,
                            diameter_um, 0.0, measured_diam_um=None, status="skipped")
                continue

            # Fire at laser-offset position
            move_and_wait(dev, 0, aligned_x + dx_nm)
            move_and_wait(dev, 1, aligned_y + dy_nm)
            time.sleep(1)
            fire_n_pulses(sig, n=cycle_count, gap_s=PULSE_GAP_S)
            time.sleep(1)

            # Capture after and evaluate
            move_and_wait(dev, 0, aligned_x)
            move_and_wait(dev, 1, aligned_y)
            after = snap_crop(orig_idx, "after", draw_circle=circle)

            exploded = difference_metric_circle(
                before, after, center=circle[:2], radius=circle[2]
            )
            elapsed = time.time() - start_time
            measured_diam_um = (2*circle[2]) * UM_PER_PX

            # LOG
            log_results(orig_idx, aligned_x, aligned_y,
                        exploded, cycle_count, diameter_um, elapsed,
                        measured_diam_um=measured_diam_um, status="done")

            # Update streak + cycle_count policy
            if exploded:
                no_explosion_streak = 0
            else:
                no_explosion_streak += 1
            if no_explosion_streak >= 5:
                cycle_count += 1
                logger.info(f"5 no-explosions in a row -> cycle_count={cycle_count}")
                no_explosion_streak = 0

    finally:
        try: sig.write("C1:OUTP OFF")
        except: pass
        try: sig.close(); rm.close()
        except: pass
        for ax in (0,1):
            try: dev.control.setControlMove(ax, False)
            except: pass
        logger.info("Hardware cleaned up.")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        logger.warning("Interrupted by user — cleaning up…")
        try: sig.write("C1:OUTP OFF")
        except: pass
        try: sig.close(); rm.close()
        except: pass
        for ax in (0,1):
            try: dev.control.setControlMove(ax, False)
            except: pass
        logger.info("Exit.")
